# Disease-gene prediction subgraph

Visualize the breast-neoplasms disease-gene subgraph and mark genes according to which benchmark methods recovered them in their top 25. A gene counts as detected only in a run where it belonged to the held-out testing set.

In [ ]:
from collections import Counter, defaultdict
from pathlib import Path
import pickle
import sys

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if not (project_root / "bioGraph").is_dir():
    raise FileNotFoundError(
        "Start Jupyter from the repository root or the notebooks directory."
    )
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import display

from bioGraph.data.loading import load_disease_genes, load_ppi_graph
from bioGraph.methods.utils import scores_to_ranking
from bioGraph.sim import validate_benchmark_results

In [ ]:
results_dir = project_root / "outputs" / "results"
artifact_path = results_dir / "results_methods_by_6disease.pkl"

if not artifact_path.is_file():
    raise FileNotFoundError(
        f"Artifact {artifact_path.name} does not exist in {results_dir}."
    )

with artifact_path.open("rb") as handle:
    benchmark_results = pickle.load(handle)
validate_benchmark_results(benchmark_results)

data_dir = project_root / "data" / "raw"
Ggen = load_ppi_graph(data_dir / "PPI202207.txt")
diseases = load_disease_genes(data_dir / "pcbi.1004120.s004.txt")
nodelist = list(Ggen.nodes())

print(f"Loaded benchmark artifact: {artifact_path}")
print(
    f"PPI: {Ggen.number_of_nodes():,} genes and "
    f"{Ggen.number_of_edges():,} interactions"
)
print(f"Saved benchmark runs: {len(benchmark_results['runs'])}")

In [ ]:
disease_name = "breast neoplasms"
top_k = 25

method_set = benchmark_results["config"]["method_set"]
runs = [
    row for row in benchmark_results["runs"]
    if row["disease"] == disease_name
]
if disease_name not in diseases:
    raise KeyError(f"Disease not found in disease-gene data: {disease_name}")
if not runs:
    raise ValueError(f"No saved benchmark runs found for {disease_name!r}.")

disease_genes = set(diseases[disease_name]) & set(Ggen.nodes())
testing_opportunities = Counter()
detections = defaultdict(Counter)

for row in runs:
    testing_genes = set(row["test_genes"]) & disease_genes
    testing_opportunities.update(testing_genes)

    for method_name in method_set:
        ranking = scores_to_ranking(
            row["scores"][method_name],
            nodelist,
            Ggen,
            row["train_genes"],
        )
        top_gene_ids = {entry["gene_id"] for entry in ranking[:top_k]}
        for gene_id in testing_genes & top_gene_ids:
            detections[gene_id][method_name] += 1

predicted_methods = {
    gene_id: {method for method in method_set if detections[gene_id][method] > 0}
    for gene_id in disease_genes
}

summary_rows = []
for gene_id in sorted(disease_genes):
    opportunity_count = testing_opportunities[gene_id]
    methods = predicted_methods[gene_id]
    row = {
        "gene_id": gene_id,
        "symbol": Ggen.nodes[gene_id].get("symbol", ""),
        "subgraph_degree": 0,
        "testing_set_runs": opportunity_count,
        "predicted_by_count": len(methods),
        "predicted_by": ", ".join(method for method in method_set if method in methods)
        or "not predicted",
    }
    for method_name in method_set:
        hit_count = detections[gene_id][method_name]
        row[f"{method_name}_hits"] = hit_count
        row[f"{method_name}_probability"] = (
            hit_count / opportunity_count if opportunity_count else np.nan
        )
    summary_rows.append(row)

prediction_summary = pd.DataFrame(summary_rows)
print(
    f"{disease_name}: {len(disease_genes)} genes in the PPI; "
    f"{sum(bool(methods) for methods in predicted_methods.values())} were recovered "
    f"by at least one method in a top-{top_k} list."
)

In [ ]:
disease_subgraph = Ggen.subgraph(disease_genes).copy()
prediction_summary["subgraph_degree"] = prediction_summary["gene_id"].map(
    dict(disease_subgraph.degree())
).fillna(0).astype(int)

# A deterministic layout is reused so every panel and PDF page is comparable.
layout = nx.spring_layout(
    disease_subgraph,
    seed=42,
    k=1.0,
    iterations=800,
    scale=4.0,
)


def separate_overlapping_nodes(positions, minimum_distance=0.55, max_iterations=500):
    """Push node centers apart until every pair meets a minimum distance."""
    separated = {node: np.asarray(position, dtype=float).copy() for node, position in positions.items()}
    nodes = list(separated)
    rng = np.random.default_rng(42)

    for _ in range(max_iterations):
        moved = False
        for left_index, left_node in enumerate(nodes):
            for right_node in nodes[left_index + 1:]:
                delta = separated[right_node] - separated[left_node]
                distance = float(np.linalg.norm(delta))
                if distance >= minimum_distance:
                    continue
                if distance < 1e-12:
                    direction = rng.normal(size=2)
                    direction /= np.linalg.norm(direction)
                else:
                    direction = delta / distance
                shift = 0.5 * (minimum_distance - distance + 1e-6) * direction
                separated[left_node] -= shift
                separated[right_node] += shift
                moved = True
        if not moved:
            break

    return separated


layout = separate_overlapping_nodes(layout)
method_count = np.array(
    [len(predicted_methods[node]) for node in disease_subgraph.nodes()], dtype=float
)

fig, ax = plt.subplots(figsize=(17, 13))
nx.draw_networkx_edges(
    disease_subgraph, layout, ax=ax, edge_color="#7A7A7A", alpha=0.75, width=2.2
)
nodes = nx.draw_networkx_nodes(
    disease_subgraph,
    layout,
    ax=ax,
    node_color=method_count,
    cmap="viridis",
    vmin=0,
    vmax=max(1, len(method_set)),
    node_size=900,
    edgecolors="black",
    linewidths=1.2,
)
labels = {node: Ggen.nodes[node].get("symbol", str(node)) for node in disease_subgraph}
nx.draw_networkx_labels(
    disease_subgraph,
    layout,
    labels,
    font_size=9,
    font_weight="bold",
    bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.72, "pad": 1.2},
    ax=ax,
)
colorbar = fig.colorbar(nodes, ax=ax, shrink=0.7)
colorbar.set_label("Number of methods that recovered the gene")
ax.set_title(
    f"{disease_name.title()} disease-gene subgraph\n"
    f"Node color = number of methods with a top-{top_k} test-set detection"
)
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
n_columns = 1
n_rows = int(np.ceil(len(method_set) / n_columns))
fig, axes = plt.subplots(n_rows, n_columns, figsize=(18, 8 * n_rows))
axes = np.atleast_1d(axes).ravel()

for ax, method_name in zip(axes, method_set):
    node_colors = [
        "#2CA02C" if method_name in predicted_methods[node] else "#D9D9D9"
        for node in disease_subgraph.nodes()
    ]
    nx.draw_networkx_edges(
        disease_subgraph, layout, ax=ax, edge_color="#777777", alpha=0.72, width=1.8
    )
    nx.draw_networkx_nodes(
        disease_subgraph,
        layout,
        ax=ax,
        node_color=node_colors,
        node_size=700,
        edgecolors="black",
        linewidths=1.0,
    )
    nx.draw_networkx_labels(
        disease_subgraph,
        layout,
        labels,
        font_size=8,
        font_weight="bold",
        bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.7, "pad": 1.0},
        ax=ax,
    )
    recovered = sum(method_name in predicted_methods[node] for node in disease_subgraph)
    ax.set_title(f"{method_name}: {recovered}/{len(disease_subgraph)} genes recovered")
    ax.axis("off")

for ax in axes[len(method_set):]:
    ax.axis("off")

legend = [
    Line2D([0], [0], marker="o", color="none", markerfacecolor="#2CA02C",
           markeredgecolor="black", markersize=10, label=f"Recovered in top {top_k}"),
    Line2D([0], [0], marker="o", color="none", markerfacecolor="#D9D9D9",
           markeredgecolor="black", markersize=10, label="Never recovered"),
]
fig.legend(handles=legend, loc="lower center", ncol=2)
fig.suptitle(
    f"Top-{top_k} test-set recovery by method: {disease_name.title()}",
    fontsize=16,
)
plt.tight_layout(rect=(0, 0.05, 1, 0.96))
plt.show()

In [ ]:
# Inspect exact methods, counts, and conditional detection probabilities per gene.
probability_columns = [f"{method}_probability" for method in method_set]
display(
    prediction_summary.sort_values(
        ["predicted_by_count", "subgraph_degree", "symbol"],
        ascending=[False, False, True],
    )[
        [
            "gene_id",
            "symbol",
            "subgraph_degree",
            "testing_set_runs",
            "predicted_by_count",
            "predicted_by",
            *probability_columns,
        ]
    ].reset_index(drop=True)
)

In [ ]:
# Export one QA page per run: seeds, detected test genes, and missed test genes.
from matplotlib.backends.backend_pdf import PdfPages
import textwrap

qa_method = "QA+"
if qa_method not in method_set:
    raise KeyError(f"{qa_method!r} is not present in the saved benchmark methods.")

figures_dir = project_root / "outputs" / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)
qa_pdf_path = figures_dir / "breast_neoplasms_QA_top25_runs.pdf"


def _format_gene_list(gene_ids, width=50):
    entries = [
        f"{Ggen.nodes[gene_id].get('symbol', gene_id)} ({gene_id})"
        for gene_id in sorted(gene_ids, key=lambda gene: Ggen.nodes[gene].get("symbol", ""))
    ]
    return textwrap.fill(", ".join(entries) or "None", width=width)


with PdfPages(qa_pdf_path) as pdf:
    for run_number, row in enumerate(runs, start=1):
        seed_genes = set(row["train_genes"]) & disease_genes
        testing_genes = set(row["test_genes"]) & disease_genes
        qa_ranking = scores_to_ranking(
            row["scores"][qa_method], nodelist, Ggen, row["train_genes"]
        )
        qa_top_gene_ids = {entry["gene_id"] for entry in qa_ranking[:top_k]}
        detected_genes = testing_genes & qa_top_gene_ids
        missed_test_genes = testing_genes - detected_genes

        node_colors = []
        node_sizes = []
        for node in disease_subgraph.nodes():
            if node in detected_genes:
                node_colors.append("#2CA02C")
                node_sizes.append(950)
            elif node in seed_genes:
                node_colors.append("#FFB000")
                node_sizes.append(850)
            elif node in missed_test_genes:
                node_colors.append("#E99696")
                node_sizes.append(700)
            else:
                node_colors.append("#D9D9D9")
                node_sizes.append(650)

        fig, (ax_graph, ax_text) = plt.subplots(
            1, 2, figsize=(17, 11), gridspec_kw={"width_ratios": [3.2, 1.25]}
        )
        nx.draw_networkx_edges(
            disease_subgraph,
            layout,
            ax=ax_graph,
            edge_color="#707070",
            alpha=0.75,
            width=2.0,
        )
        nx.draw_networkx_nodes(
            disease_subgraph,
            layout,
            ax=ax_graph,
            node_color=node_colors,
            node_size=node_sizes,
            edgecolors="black",
            linewidths=1.1,
        )
        nx.draw_networkx_labels(
            disease_subgraph,
            layout,
            labels,
            font_size=8,
            font_weight="bold",
            bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.72, "pad": 1.0},
            ax=ax_graph,
        )
        ax_graph.set_title(
            f"QA run {run_number} (split seed {row['seed']})\n"
            f"{len(detected_genes)}/{len(testing_genes)} testing genes detected in top {top_k}",
            fontsize=15,
        )
        ax_graph.axis("off")

        legend = [
            Line2D([0], [0], marker="o", color="none", markerfacecolor="#FFB000",
                   markeredgecolor="black", markersize=12, label="Training seed"),
            Line2D([0], [0], marker="o", color="none", markerfacecolor="#2CA02C",
                   markeredgecolor="black", markersize=12, label=f"Detected test gene (top {top_k})"),
            Line2D([0], [0], marker="o", color="none", markerfacecolor="#E99696",
                   markeredgecolor="black", markersize=12, label="Missed test gene"),
        ]
        ax_graph.legend(handles=legend, loc="upper left", frameon=True)

        details = (
            f"TRAINING SEEDS ({len(seed_genes)})\n"
            f"{_format_gene_list(seed_genes)}\n\n"
            f"DETECTED TEST GENES ({len(detected_genes)})\n"
            f"{_format_gene_list(detected_genes)}\n\n"
            f"MISSED TEST GENES ({len(missed_test_genes)})\n"
            f"{_format_gene_list(missed_test_genes)}"
        )
        ax_text.text(0.0, 1.0, details, va="top", ha="left", fontsize=10, linespacing=1.35)
        ax_text.axis("off")

        fig.suptitle(
            f"{disease_name.title()} — QA top-{top_k} predictions",
            fontsize=18,
            y=0.99,
        )
        plt.tight_layout(rect=(0, 0, 1, 0.97))
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

print(f"Saved {len(runs)} QA run pages to: {qa_pdf_path}")
